# MREN 403 - EMG Signal Classification

## INIT

Setup

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Copy project to runtime

In [2]:
%cd "/content/drive/MyDrive/Colab Notebooks"
!cp -r "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" /content/

/content/drive/MyDrive/Colab Notebooks


Check numer of CPU cores & GPU Info

In [ ]:
import os
os.cpu_count()

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Fri Feb 20 23:48:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

INIT Model

In [ ]:
!python model.py

Model initialized with 6 channels. Feature Mode: NONE


Check Dataset Format

In [ ]:
import scipy.io as sio
import numpy as np

# Replace this with the path to one of your realMove .mat files
file_path = "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TRAIN/EMG_session1_sub10_multigrasp_realMove.mat"

print(f"Inspecting: {file_path}")
try:
    # Load the file exactly as the dataset script does
    mat_data = sio.loadmat(file_path, squeeze_me=True, struct_as_record=False)

    print("\n--- Root Keys ---")
    print([k for k in mat_data.keys() if not k.startswith('__')])

    for struct_name in ['dat', 'mrk']:
        if struct_name in mat_data:
            obj = mat_data[struct_name]
            print(f"\n--- Structure: '{struct_name}' ---")
            print(f"Type: {type(obj)}")

            # Check if it is wrapped in a numpy array
            if isinstance(obj, np.ndarray):
                print(f"Array Shape: {obj.shape}")
                print(f"Array Dtype: {obj.dtype}")
                # Try to extract the first item if it's an array of objects
                if obj.size > 0:
                    obj = obj.item() if obj.ndim == 0 else obj[0]
                    print(f"Unwrapped Type: {type(obj)}")

            # Print the actual MATLAB fields
            if hasattr(obj, '_fieldnames'):
                print(f"Fields: {obj._fieldnames}")
            else:
                print("No standard MATLAB fields found.")
        else:
            print(f"\n--- Structure: '{struct_name}' NOT FOUND ---")

except Exception as e:
    print(f"Failed to load or inspect file: {e}")

Inspecting: /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TRAIN/EMG_session1_sub10_multigrasp_realMove.mat

--- Root Keys ---
['mrk', 'mnt', 'nfo', 'ch1', 'ch2', 'ch3', 'ch4', 'ch5', 'ch6', 'ch7', 'dat']

--- Structure: 'dat' ---
Type: <class 'scipy.io.matlab._mio5_params.mat_struct'>
Fields: ['clab', 'fs', 'title', 'file', 'resolution']

--- Structure: 'mrk' ---
Type: <class 'scipy.io.matlab._mio5_params.mat_struct'>
Fields: ['pos', 'toe', 'fs', 'y', 'className', 'misc']


INIT Dataset

In [ ]:
!python dataset.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --plot_avg \
  --plot_individual

Extracting real data to plot average EMG envelope for 6 electrodes...
Plot saved successfully as 'average_emg_6_electrodes.png'.
Extracting real data to plot individual EMG signals for 6 electrodes...
Plot saved successfully as 'individual_emg_6_electrodes.png'.


## All nodes model

Train on 6 electrodes - Using Classification with Regression

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_6ch.pth" \
  --task multi \
  --epochs 100 \
  --lr 0.001 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: MULTI
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Epoch 001/100 | Time: 189.84s
  Train -> Total: 3.4821 | Cls: 2.0872 | Reg: 0.0139 | Acc: 27.36% (167.06s)
  Val   -> Total: 3.5143 | Cls: 2.0905 | Reg: 0.0142 | Acc: 29.52% (22.77s)
  >>> Best multi model saved to emg_model_6ch_multi.pth
Epoch 002/100 | Time: 191.39s
  Train -> Total: 3.2273 | Cls: 1.9556 | Reg: 0.0127 | Acc: 32.64% (168.65s)
  Val   -> Total: 3.5915 | Cls: 2.1392 | Reg: 0.0145 | Acc: 30.10% (22.74s)
  >>> No improvement for 1 epoch(s).
Epoch 003/100 | Time: 191.86s
  Train -> Total: 3.1659 | Cls: 1.9192 | Reg: 0.0125 | Acc: 34.10% (168.97s)
  Val   -> Total: 3.7035 | Cls: 2.2245 | Reg: 0.0148 | Acc: 29.33% (22.89s)
  >>> No improvement for 2 epoch(s).
Epoch 004/100 | Time: 193.84s
  Train -> To

Train on 6 electrodes - Using only Classification

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_6ch_classification.pth" \
  --task classification \
  --epochs 100 \
  --lr 0.001 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: CLASSIFICATION
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Epoch 001/100 | Time: 187.40s
  Train -> Total: 2.1460 | Cls: 2.1460 | Reg: 0.1054 | Acc: 24.32% (164.78s)
  Val   -> Total: 2.1175 | Cls: 2.1175 | Reg: 0.0975 | Acc: 29.26% (22.62s)
  >>> Best classification model saved to emg_model_6ch_classification_classification.pth
Epoch 002/100 | Time: 187.36s
  Train -> Total: 1.9311 | Cls: 1.9311 | Reg: 0.1018 | Acc: 33.54% (164.43s)
  Val   -> Total: 2.2058 | Cls: 2.2058 | Reg: 0.1007 | Acc: 29.48% (22.93s)
  >>> No improvement for 1 epoch(s).
Epoch 003/100 | Time: 185.93s
  Train -> Total: 1.8930 | Cls: 1.8930 | Reg: 0.1012 | Acc: 34.97% (163.14s)
  Val   -> Total: 2.2090 | Cls: 2.2090 | Reg: 0.1024 | Acc: 30.13% (22.79s)
  >>> No improvement for 2 epoch(s).
E

Train on 6 electrodes - Using Classification with Regression, activating Time-Domain Features

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_6ch.pth" \
  --model_type adaptive \
  --task multi \
  --epochs 100 \
  --lr 0.005 \
  --scaling constant \
  --feature_ext td \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: MULTI | Model: ADAPTIVE
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: CONSTANT | Feature Ext: TD
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: CONSTANT | Feature Ext: TD
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
Epoch 0

Test on 6 electrodes

In [ ]:
!python test.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --weights "/content/drive/MyDrive/Colab Notebooks/emg_model_6ch_multi.pth" \
  --batch_size 16 \
  --electrodes 0 1 2 3 4 5

Testing on device: cuda
Routing Training Data -> /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TRAIN
Routing Testing Data  -> /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TEST
Searching for .mat files in /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TRAIN...
Extracted 67107 total trials. Balancing to ~2991 trials per class.
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Searching for .mat files in /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TEST...
Extracted 16500 total trials. Balancing to ~750 trials per class.
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
>>> Model weights loaded successfully.
Starting inference evaluation...

--- Testing Statistics ---
Total Testing Time: 129.3947s
Inference Speed Min: 0.0747 ms/signal
Inference Speed Max: 25.265

Visualize 6 electrodes

In [ ]:
!python visualize.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --weights "/content/drive/MyDrive/Colab Notebooks/emg_model_6ch_multi.pth" \
  --electrodes 0 1 2 3 4 5 \
  --batch_size 256 \
  --num_workers 4

Visualizing on device: cuda
Routing Training Data -> /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TRAIN
Routing Testing Data  -> /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TEST
Searching for .mat files in /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TRAIN...
Extracted 67107 total trials. Balancing to ~2991 trials per class.
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Searching for .mat files in /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TEST...
Extracted 16500 total trials. Balancing to ~750 trials per class.
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
>>> Model weights loaded successfully.
Running batched inference (Batch Size: 256)...
/content/drive/MyDrive/Colab Notebooks/visualize.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)`

## NEW MODELS

INIT Model

In [ ]:
!python model.py

INIT Dataset

In [ ]:
!python dataset.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --plot_avg \
  --plot_individual

Train on 6 electrodes - Using 1D-CNN

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_6ch.pth" \
  --model_type multiscale_cnn \
  --task multi \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: MULTI | Model: MULTISCALE_CNN
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
E

Train

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_6ch.pth" \
  --task multi \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: MULTI | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
Epoch 0

Train but classification

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_6ch.pth" \
  --task classification \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: CLASSIFICATION | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp)

Train on 1

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/1subject" \
  --model_path "1sub_emg_model_6ch.pth" \
  --task multi \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: MULTI | Model: TCN_LSTM
Dataset initialized: 1200 trials mapped to 92400 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 600 trials mapped to 46200 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
Epoch 001/10

Train on 1 just reaching

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/1subject_reaching" \
  --model_path "1subReach_emg_model_6ch.pth" \
  --task classification \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: CLASSIFICATION | Model: TCN_LSTM
Dataset initialized: 378 trials mapped to 29106 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 189 trials mapped to 14553 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
Epoc

# FINAL

Train

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_6ch.pth" \
  --task multi \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: MULTI | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
Epoch 0

Test

In [ ]:
!python test.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --weights "emg_model_6ch_tcn_lstm_multi.pth" \
  --model_type tcn_lstm \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5

Testing on device: cuda | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
>>> Model weights loaded successfully.
Starting inference evaluation...

--- Testing Statistics ---
Total Testing Time: 21.3985s
Inference Speed Mean: 0.0154 ms/signal
Data saved to test_results_tcn_lstm.json


Visualize

In [ ]:
!python visualize.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --weights "emg_model_6ch_tcn_lstm_multi.pth" \
  --model_type tcn_lstm \
  --scaling zscore \
  --feature_ext none \
  --electrodes 0 1 2 3 4 5

Visualizing on device: cuda | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
>>> Model weights loaded successfully.
Running batched inference (Batch Size: 256)...
/content/drive/MyDrive/Colab Notebooks/visualize.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
Concatenating batches...

--- Performance Metrics (R^2) ---
All Results   - Min: -2.6160, Max: 1.0000, Mean: 0.8142, StDev: 0.3802
Saved Top 4 Best visualizations.
Saved Bottom 4 Worst visualizations.


Train on 4

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_4ch.pth" \
  --task multi \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: MULTI | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
Epoch 0

Test on 4

In [ ]:
!python test.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --weights "emg_model_4ch_tcn_lstm_multi.pth" \
  --model_type tcn_lstm \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 2 3 4 5

Testing on device: cuda | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
>>> Model weights loaded successfully.
Starting inference evaluation...

--- Testing Statistics ---
Total Testing Time: 20.3185s
Inference Speed Mean: 0.0151 ms/signal
Data saved to test_results_tcn_lstm.json


Visualize on 4

In [ ]:
!python visualize.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --weights "emg_model_4ch_tcn_lstm_multi.pth" \
  --model_type tcn_lstm \
  --scaling zscore \
  --feature_ext none \
  --electrodes 2 3 4 5

Visualizing on device: cuda | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
>>> Model weights loaded successfully.
Running batched inference (Batch Size: 256)...
/content/drive/MyDrive/Colab Notebooks/visualize.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
Concatenating batches...

--- Performance Metrics (R^2) ---
All Results   - Min: -2.6320, Max: 1.0000, Mean: 0.7928, StDev: 0.4533
Saved Top 4 Best visualizations.
Saved Bottom 4 Worst visualizations.


Train on 2

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "emg_model_2ch.pth" \
  --task multi \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 4 5 \
  --use_amp

Using device: cuda
Training Mode: MULTI | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
Epoch 0

Test on 2

In [ ]:
!python test.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --weights "emg_model_2ch_tcn_lstm_multi.pth" \
  --model_type tcn_lstm \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 4 5

Testing on device: cuda | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
>>> Model weights loaded successfully.
Starting inference evaluation...

--- Testing Statistics ---
Total Testing Time: 19.3206s
Inference Speed Mean: 0.0144 ms/signal
Data saved to test_results_tcn_lstm.json


Visualize on 2

In [ ]:
!python visualize.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --weights "emg_model_2ch_tcn_lstm_multi.pth" \
  --model_type tcn_lstm \
  --scaling zscore \
  --feature_ext none \
  --electrodes 4 5

Visualizing on device: cuda | Model: TCN_LSTM
Dataset initialized: 35814 trials mapped to 2757678 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 9000 trials mapped to 693000 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
>>> Model weights loaded successfully.
Running batched inference (Batch Size: 256)...
/content/drive/MyDrive/Colab Notebooks/visualize.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
Concatenating batches...

--- Performance Metrics (R^2) ---
All Results   - Min: -2.3461, Max: 1.0000, Mean: 0.7581, StDev: 0.5165
Saved Top 4 Best visualizations.
Saved Bottom 4 Worst visualizations.


Train w\ 1 subject

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/1subject_reaching" \
  --model_path "1sub_emg_model_6ch.pth" \
  --task classification \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --use_amp

Using device: cuda
Training Mode: CLASSIFICATION | Model: TCN_LSTM
Dataset initialized: 378 trials mapped to 29106 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 189 trials mapped to 14553 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:38: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:111: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
Epoc

Test w\ 1 subject

In [ ]:
!python test.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/1subject_reaching" \
  --weights "1sub_emg_model_6ch_tcn_lstm_multi.pth" \
  --model_type tcn_lstm \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5

Testing on device: cuda | Model: TCN_LSTM
Dataset initialized: 378 trials mapped to 29106 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 189 trials mapped to 14553 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
>>> Model weights loaded successfully.
Starting inference evaluation...

--- Testing Statistics ---
Total Testing Time: 0.9250s
Inference Speed Mean: 0.0391 ms/signal
Data saved to test_results_tcn_lstm.json


Visualize w\ 1 subject

In [ ]:
!python visualize.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/1subject_reaching" \
  --weights "1sub_emg_model_6ch_tcn_lstm_multi.pth" \
  --model_type tcn_lstm \
  --scaling zscore \
  --feature_ext none \
  --electrodes 0 1 2 3 4 5

Visualizing on device: cuda | Model: TCN_LSTM
Dataset initialized: 378 trials mapped to 29106 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 189 trials mapped to 14553 on-the-fly windows.
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
>>> Model weights loaded successfully.
Running batched inference (Batch Size: 256)...
/content/drive/MyDrive/Colab Notebooks/visualize.py:63: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
Concatenating batches...

--- Performance Metrics (R^2) ---
All Results   - Min: -0.0088, Max: 0.9999, Mean: 0.9213, StDev: 0.1203
Saved Top 4 Best visualizations.
Saved Bottom 4 Worst visualizations.


## Check Signals Variance

In [ ]:
!pip install fastdtw

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fastdtw: filename=fastdtw-0.3.4-cp312-cp312-linux_x86_64.whl size=567861 sha256=79908abc293baec9c3cf1952e63bd6ac4eeae95b1691289a50cfe79b4eeecf1e
  Stored in directory: /root/.cache/pip/wheels/ab/d0/26/b82cb0f49ae73e5e6bba4e8462fff2c9851d7bd2ec64f8891e
Successfully built fastdtw


In [ ]:
!python signals.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --electrodes 0 1 2 3 4 5 \
  --option all


[1/7] Scanning /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove for all .mat files...
  -> Found 224 files. Beginning extraction...
  -> Parsing file 224/224...
  -> Data extraction complete. Balancing classes to prevent overweighting...
  -> Target trial count per class set to: 3741

  -> Final balanced 500-sample trials by class:
     [Rest]: 3741 trials
     [Arm-reaching: Forward]: 3741 trials
     [Arm-reaching: Backward]: 3741 trials
     [Arm-reaching: Up]: 3741 trials
     [Arm-reaching: Down]: 3741 trials
     [Arm-reaching: Left]: 3741 trials
     [Arm-reaching: Right]: 3741 trials
     [Wrist-twisting: Pronation]: 3703 trials
     [Wrist-twisting: Supination]: 3701 trials
     [Hand-grasping: Card]: 3741 trials
     [Hand-grasping: Ball]: 3741 trials
     [Hand-grasping: Cup]: 3741 trials

[2/7] Generating Time-Frequency Spectrograms per Electrode...
  -> Processing Spectrograms for Channel 0...
  -> Processing Spectrograms for Channel 1...
  -> Processing 

## Train on 11 movmements, no rest

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_path "11mov_emg_model_6ch.pth" \
  --task multi \
  --model_type tcn_lstm \
  --epochs 100 \
  --lr 0.002 \
  --scaling zscore \
  --feature_ext none \
  --batch_size 256 \
  --num_workers 8 \
  --electrodes 0 1 2 3 4 5 \
  --exclude_rest \
  --use_amp

Using device: cuda
Training Mode: MULTI | Model: TCN_LSTM | Classes: 11-Class (No Rest)
Dataset initialized: 32823 trials mapped to 2527371 on-the-fly windows. [11-Class (No Rest)]
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
Dataset initialized: 8250 trials mapped to 635250 on-the-fly windows. [11-Class (No Rest)]
Pipeline -> Scaling: ZSCORE | Feature Ext: NONE
/content/drive/MyDrive/Colab Notebooks/train.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)

diagnose

In [ ]:
!python diagnose_raw_emg.py --file "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TEST/EMG_session1_sub20_reaching_realMove.mat"

Loading raw session file: /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove/EMG_ConvertedData_realMove_TEST/EMG_session1_sub20_reaching_realMove.mat
Total Session Array Shape: (8449600, 6) (Samples, Channels)
Found 600 total triggers.

--- Diagnostic Extraction ---
Hardware Sampling Rate: ~120 Hz
Extracting exactly 480 samples (4 seconds) per movement.
Saved -> diagnostic_raw_voltage.png

Calculating FFT Frequency Spectrum...
Saved -> diagnostic_fft_spectrum.png


# NEW NEW

In [ ]:
!python signals.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --option all


[1/7] Scanning /content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove for all .mat files...
  -> Found 224 files. Beginning extraction...
  -> Parsing file 224/224...
  -> Data extraction complete. Filter Applied: True (20.0Hz). Balancing classes...
  -> Target trial count per class set to: 3741

  -> Final balanced 500-sample trials by class:
     [Rest]: 3741 trials
     [Arm-reaching: Forward]: 3741 trials
     [Arm-reaching: Backward]: 3741 trials
     [Arm-reaching: Up]: 3741 trials
     [Arm-reaching: Down]: 3741 trials
     [Arm-reaching: Left]: 3741 trials
     [Arm-reaching: Right]: 3741 trials
     [Wrist-twisting: Pronation]: 3703 trials
     [Wrist-twisting: Supination]: 3701 trials
     [Hand-grasping: Card]: 3741 trials
     [Hand-grasping: Ball]: 3741 trials
     [Hand-grasping: Cup]: 3741 trials

[2/7] Generating Time-Frequency Spectrograms per Electrode...
  -> Processing Spectrograms for Channel 0...
  -> Processing Spectrograms for Channel 1...
  -> Proce

NEW NEW Train

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --model_type adaptive \
  --task multi \
  --epochs 100 \
  --scaling constant \
  --lr 0.001\
  --batch_size 256 \
  --feature_ext td \
  --exclude_rest

Using device: cuda
Training Mode: MULTI | Model: ADAPTIVE
Dataset initialized: 32823 trials mapped to 2527371 on-the-fly windows. [11-Class (No Rest)]
Pipeline -> Scaling: CONSTANT | Feature Ext: TD | Filter: True (20.0Hz)
Dataset initialized: 8250 trials mapped to 635250 on-the-fly windows. [11-Class (No Rest)]
Pipeline -> Scaling: CONSTANT | Feature Ext: TD | Filter: True (20.0Hz)
/content/drive/MyDrive/Colab Notebooks/train.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocas

In [ ]:
!python train.py \
  --data_root "/content/drive/MyDrive/Colab Notebooks/1subject" \
  --model_type adaptive \
  --task multi \
  --scaling constant \
  --lr 0.001\
  --epochs 100 \
  --batch_size 64 \
  --feature_ext td \
  --exclude_rest

Using device: cuda
Training Mode: MULTI | Model: ADAPTIVE
Dataset initialized: 1100 trials mapped to 84700 on-the-fly windows. [11-Class (No Rest)]
Pipeline -> Scaling: CONSTANT | Feature Ext: TD | Filter: True (20.0Hz)
Dataset initialized: 550 trials mapped to 42350 on-the-fly windows. [11-Class (No Rest)]
Pipeline -> Scaling: CONSTANT | Feature Ext: TD | Filter: True (20.0Hz)
/content/drive/MyDrive/Colab Notebooks/train.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
/content/drive/MyDrive/Colab Notebooks/train.py:71: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=args.use_amp):
/content/drive/MyDrive/Colab Notebooks/train.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cu

# SIMULATE

In [8]:
!python simulate.py \
  --weights "/content/drive/MyDrive/Colab Notebooks/11mov_emg_model_6ch_tcn_lstm_multi.pth" \
  --data_root "/content/drive/MyDrive/Colab Notebooks/EMG_ConvertedData_realMove" \
  --movement "Arm-reaching: Right" \
  --model_type adaptive \
  --feature_ext td \
  --exclude_rest

Simulation Device: cuda | Model: ADAPTIVE
Dataset initialized: 8250 trials mapped to 635250 on-the-fly windows. [11-Class (No Rest)]
Pipeline -> Scaling: CONSTANT | Feature Ext: TD | Filter: True (20.0Hz)
Selected Trial Index 1178 out of 750 available 'Arm-reaching: Right' trials.
Error loading model weights: [Errno 2] No such file or directory: '/content/drive/MyDrive/Colab Notebooks/11mov_emg_model_6ch_tcn_lstm_multi.pth'


Viz best model